# F3-Net: Frequency-Aware Training
Training an AI-generated image detector using frequency-domain features (FFT magnitude spectrum).

## 1. Setup

In [11]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
from tqdm import tqdm
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


## 2. Config

In [12]:
CFG = {
    # ── Paths ──────────────────────────────────────────────────────────────────
    "dataset_path": "/workspace/datasets/ai-gen/mnt/c/Development/ai-gen-detection/shard_0/shard_0",

    # ── Training ───────────────────────────────────────────────────────────────
    "batch_size":   24,
    "val_batch_size": 48,       # no gradients → can afford 2× batch
    "epochs":       12,
    "lr":           1e-4,
    "weight_decay": 1e-4,
    "num_workers":  8,

    # ── Image ──────────────────────────────────────────────────────────────────
    "img_size":     256,        # spatial resolution fed to the network
    "val_split":    0.1,
    "seed":         SEED,
}

print("Config:")
for k, v in CFG.items():
    print(f"  {k}: {v}")

Config:
  dataset_path: /workspace/datasets/ai-gen/mnt/c/Development/ai-gen-detection/shard_0/shard_0
  batch_size: 24
  val_batch_size: 48
  epochs: 12
  lr: 0.0001
  weight_decay: 0.0001
  num_workers: 8
  img_size: 256
  val_split: 0.1
  seed: 42


## 3. Transforms
Using `albumentations` for both spatial and compression augmentations.
`ImageCompression` simulates JPEG artefacts at varying quality levels, which helps the network generalise across social-media-recompressed images.

In [14]:
IMG_SIZE = CFG["img_size"]

# ImageNet statistics — used for the RGB stream
_MEAN = (0.485, 0.456, 0.406)
_STD  = (0.229, 0.224, 0.225)

train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.ImageCompression(quality_range=(50, 99), p=0.5),
    A.Normalize(mean=_MEAN, std=_STD),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=_MEAN, std=_STD),
    ToTensorV2(),
])

print("Train transform:")
for t in train_transform.transforms:
    print(f"  {t.__class__.__name__}")

print("\nVal transform:")
for t in val_transform.transforms:
    print(f"  {t.__class__.__name__}")

Train transform:
  Resize
  HorizontalFlip
  ImageCompression
  Normalize
  ToTensorV2

Val transform:
  Resize
  Normalize
  ToTensorV2


## 4. Dataset Class
Each sample returns **two tensors**:
- `rgb_tensor` — the spatially-normalised image `(3, H, W)`
- `fft_tensor` — the log-scaled FFT magnitude spectrum `(1, H, W)`, computed on the grayscale image before any normalisation

The FFT branch captures periodic artefacts (GAN upsampling grids, diffusion noise patterns) that are often invisible in the spatial domain.

In [15]:
class FFTDataset(Dataset):
    """
    Loads images from a shard directory (same layout as AIGenDetDataset).

    Returns
    -------
    rgb_tensor : FloatTensor  (3, H, W)  — spatially augmented + normalised
    fft_tensor : FloatTensor  (1, H, W)  — log FFT magnitude spectrum
    label      : LongTensor   scalar
    """

    def __init__(self, shard_path: str, transform=None):
        self.shard_path = shard_path
        self.transform  = transform
        self.labels     = pd.read_csv(os.path.join(shard_path, "labels.csv"))
        print(f"[FFTDataset] Loaded {len(self.labels)} samples from {shard_path}")

    def __len__(self) -> int:
        return len(self.labels)

    @staticmethod
    def _fft_magnitude(pil_img: Image.Image, size: int) -> torch.Tensor:
        """
        Convert a PIL RGB image to a (1, size, size) log-magnitude FFT tensor.

        Steps
        -----
        1. Convert to grayscale float32 in [0, 1].
        2. Resize to (size, size) so the spectrum has a fixed shape.
        3. Compute 2-D DFT, shift the zero-frequency component to the centre.
        4. Take log(1 + |F|) to compress the dynamic range.
        5. Min-max normalise to [0, 1].
        """
        gray = np.array(
            pil_img.convert("L").resize((size, size), Image.BILINEAR),
            dtype=np.float32,
        ) / 255.0

        fft        = np.fft.fft2(gray)
        fft_shift  = np.fft.fftshift(fft)
        magnitude  = np.log1p(np.abs(fft_shift))           # log-scale

        # Min-max normalise
        mag_min, mag_max = magnitude.min(), magnitude.max()
        if mag_max - mag_min > 1e-8:
            magnitude = (magnitude - mag_min) / (mag_max - mag_min)

        # Explicit float32 cast: np.fft produces float64 intermediates
        return torch.from_numpy(magnitude.astype(np.float32)).unsqueeze(0)  # (1, H, W)

    def __getitem__(self, idx: int):
        row      = self.labels.iloc[idx]
        img_path = os.path.join(self.shard_path, "images", row["image_name"])
        pil_img  = Image.open(img_path).convert("RGB")

        # ── FFT branch (computed on the raw PIL image, before spatial augment) ─
        fft_tensor = self._fft_magnitude(pil_img, CFG["img_size"])

        # ── RGB branch ─────────────────────────────────────────────────────────
        if self.transform is not None:
            rgb_np     = np.array(pil_img)
            augmented  = self.transform(image=rgb_np)
            rgb_tensor = augmented["image"]             # ToTensorV2 → (3, H, W)
        else:
            rgb_tensor = transforms.ToTensor()(pil_img)

        label = torch.tensor(row["label"], dtype=torch.long)

        return rgb_tensor, fft_tensor, label


# ── Smoke-test ─────────────────────────────────────────────────────────────────
if os.path.exists(CFG["dataset_path"]):
    _ds = FFTDataset(CFG["dataset_path"], transform=val_transform)
    _rgb, _fft, _lbl = _ds[0]
    print(f"RGB tensor shape : {tuple(_rgb.shape)}")
    print(f"FFT tensor shape : {tuple(_fft.shape)}")
    print(f"FFT value range  : [{_fft.min():.4f}, {_fft.max():.4f}]")
    print(f"Label            : {_lbl.item()}")
    del _ds, _rgb, _fft, _lbl
else:
    print(f"[INFO] Dataset path not found locally — skipping smoke-test.")

[FFTDataset] Loaded 50000 samples from /workspace/datasets/ai-gen/mnt/c/Development/ai-gen-detection/shard_0/shard_0
RGB tensor shape : (3, 256, 256)
FFT tensor shape : (1, 256, 256)
FFT value range  : [0.0000, 1.0000]
Label            : 0


## 5. DataLoaders

In [16]:
full_dataset = FFTDataset(CFG["dataset_path"], transform=None)

indices = list(range(len(full_dataset)))
train_idx, val_idx = train_test_split(
    indices,
    test_size=CFG["val_split"],
    random_state=CFG["seed"],
    stratify=full_dataset.labels["label"],
)
print(f"Train samples: {len(train_idx)} | Val samples: {len(val_idx)}")


class _SplitDataset(Dataset):
    """Wraps a subset of FFTDataset and applies per-split transforms."""

    def __init__(self, base_dataset: FFTDataset, indices: list, transform=None):
        self.base      = base_dataset
        self.indices   = indices
        self.transform = transform

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, idx: int):
        real_idx = self.indices[idx]
        row      = self.base.labels.iloc[real_idx]
        img_path = os.path.join(self.base.shard_path, "images", row["image_name"])
        pil_img  = Image.open(img_path).convert("RGB")

        fft_tensor = FFTDataset._fft_magnitude(pil_img, CFG["img_size"])

        if self.transform is not None:
            rgb_np     = np.array(pil_img)
            augmented  = self.transform(image=rgb_np)
            rgb_tensor = augmented["image"]
        else:
            rgb_tensor = transforms.ToTensor()(pil_img)

        label = torch.tensor(row["label"], dtype=torch.long)
        return rgb_tensor, fft_tensor, label


train_split = _SplitDataset(full_dataset, train_idx, transform=train_transform)
val_split   = _SplitDataset(full_dataset, val_idx,   transform=val_transform)

_loader_kwargs = dict(
    num_workers=CFG["num_workers"],
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
)

train_loader = DataLoader(
    train_split,
    batch_size=CFG["batch_size"],
    shuffle=True,
    **_loader_kwargs,
)

val_loader = DataLoader(
    val_split,
    batch_size=CFG["val_batch_size"],
    shuffle=False,
    **_loader_kwargs,
)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

[FFTDataset] Loaded 50000 samples from /workspace/datasets/ai-gen/mnt/c/Development/ai-gen-detection/shard_0/shard_0
Train samples: 45000 | Val samples: 5000
Train batches: 1875 | Val batches: 105


## 6. F3Net Model
A lightweight dual-stream CNN that processes **RGB** and **FFT magnitude** in parallel before fusing them for binary classification.

Stream architecture (shared design):
- Three conv blocks, each: `Conv2d → BN → ReLU → MaxPool`
- Channel progression: `3→32→64→128` (RGB) and `1→32→64→128` (FFT)
- Both streams end with **Global Average Pooling** → 128-d vector
- Fused 256-d vector → linear head → **1 output logit** (BCEWithLogitsLoss)

In [17]:
def _conv_block(in_ch: int, out_ch: int, pool: bool = True) -> nn.Sequential:
    """Conv2d(3×3) → BN → ReLU [ → MaxPool(2×2) ]."""
    layers: list[nn.Module] = [
        nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
    ]
    if pool:
        layers.append(nn.MaxPool2d(2, 2))
    return nn.Sequential(*layers)


class F3Net(nn.Module):
    """
    Frequency-Focused Forgery Detection Network (simplified).

    Dual-stream architecture:
      • RGB stream   — captures spatial/texture artefacts
      • FFT stream   — captures frequency-domain artefacts

    Both streams share the same conv-block topology; their 128-d GAP
    embeddings are concatenated and fed into a single linear head that
    outputs one raw logit (compatible with BCEWithLogitsLoss).
    """

    def __init__(self, dropout: float = 0.3):
        super().__init__()

        # ── RGB stream (3 → 32 → 64 → 128) ───────────────────────────────────
        self.rgb_stream = nn.Sequential(
            _conv_block(3,   32),   # 256 → 128
            _conv_block(32,  64),   # 128 →  64
            _conv_block(64, 128),   #  64 →  32
        )

        # ── FFT stream (1 → 32 → 64 → 128) ───────────────────────────────────
        self.fft_stream = nn.Sequential(
            _conv_block(1,   32),
            _conv_block(32,  64),
            _conv_block(64, 128),
        )

        # ── Global Average Pooling ─────────────────────────────────────────────
        self.gap = nn.AdaptiveAvgPool2d(1)   # (B, 128, H, W) → (B, 128, 1, 1)

        # ── Fusion head ───────────────────────────────────────────────────────
        # Concatenated embedding: 128 (RGB) + 128 (FFT) = 256-d
        self.head = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(256, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout / 2),
            nn.Linear(64, 1),       # single logit
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, rgb: torch.Tensor, fft: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        rgb : (B, 3, H, W)
        fft : (B, 1, H, W)

        Returns
        -------
        logits : (B, 1)
        """
        rgb_feat = self.gap(self.rgb_stream(rgb)).flatten(1)   # (B, 128)
        fft_feat = self.gap(self.fft_stream(fft)).flatten(1)   # (B, 128)
        fused    = torch.cat([rgb_feat, fft_feat], dim=1)       # (B, 256)
        return self.head(fused)                                  # (B, 1)

## 7. Model Setup

In [18]:
model = F3Net(dropout=0.3).to(device)

# ── Parameter count ───────────────────────────────────────────────────────────
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# ── Forward-pass shape check ──────────────────────────────────────────────────
_rgb_probe = torch.zeros(2, 3, CFG["img_size"], CFG["img_size"], device=device)
_fft_probe = torch.zeros(2, 1, CFG["img_size"], CFG["img_size"], device=device)
with torch.no_grad():
    _out = model(_rgb_probe, _fft_probe)
print(f"Output shape (batch=2): {tuple(_out.shape)}")
del _rgb_probe, _fft_probe, _out

Total parameters    : 202,881
Trainable parameters: 202,881
Output shape (batch=2): (2, 1)


## 8. Optimizer, Scheduler & Loss

In [19]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG["lr"],
    weight_decay=CFG["weight_decay"],
)

# Cosine decay from lr → eta_min over all training epochs
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CFG["epochs"],
    eta_min=1e-6,
)

criterion = nn.BCEWithLogitsLoss()

# AMP scaler — disabled automatically when CUDA is unavailable
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

print(f"Optimizer : AdamW  (lr={CFG['lr']}, weight_decay={CFG['weight_decay']})")
print(f"Scheduler : CosineAnnealingLR  (T_max={CFG['epochs']}, eta_min=1e-6)")
print(f"Loss      : BCEWithLogitsLoss")
print(f"AMP scaler enabled: {scaler.is_enabled()}")

Optimizer : AdamW  (lr=0.0001, weight_decay=0.0001)
Scheduler : CosineAnnealingLR  (T_max=12, eta_min=1e-6)
Loss      : BCEWithLogitsLoss
AMP scaler enabled: True


## 9. Training & Validation Functions

In [20]:
def train_one_epoch(loader, log_interval: int = 50) -> float:
    """
    One full pass over the training set with AMP + GradScaler.

    Parameters
    ----------
    loader       : DataLoader yielding (rgb, fft, label)
    log_interval : print running loss every N batches

    Returns
    -------
    mean loss over all batches
    """
    model.train()
    running_loss = 0.0

    for batch_idx, (rgb, fft, labels) in enumerate(tqdm(loader, desc="  train", leave=False)):
        rgb    = rgb.to(device, non_blocking=True)
        fft    = fft.to(device, non_blocking=True)
        labels = labels.float().unsqueeze(1).to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=scaler.is_enabled()):
            logits = model(rgb, fft)
            loss   = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()

        if (batch_idx + 1) % log_interval == 0:
            avg = running_loss / (batch_idx + 1)
            tqdm.write(f"    batch {batch_idx + 1:4d}/{len(loader)} | loss {avg:.5f}")

    return running_loss / len(loader)


@torch.no_grad()
def validate(loader) -> tuple[float, float]:
    """
    Evaluate on a loader, returning (accuracy, roc_auc).

    Returns
    -------
    accuracy : fraction of correct binary predictions (threshold = 0.5)
    roc_auc  : area under the ROC curve
    """
    model.eval()
    all_probs  = []
    all_labels = []

    for rgb, fft, labels in tqdm(loader, desc="  val  ", leave=False):
        rgb = rgb.to(device, non_blocking=True)
        fft = fft.to(device, non_blocking=True)

        with torch.amp.autocast("cuda", enabled=scaler.is_enabled()):
            logits = model(rgb, fft)

        probs = torch.sigmoid(logits).squeeze(1)   # (B,)
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.numpy())

    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)

    preds    = (all_probs >= 0.5).astype(int)
    accuracy = (preds == all_labels).mean()
    roc_auc  = roc_auc_score(all_labels, all_probs)

    return float(accuracy), float(roc_auc)

## 10. Epoch Training (12 epochs)

In [21]:
CKPT_DIR   = os.path.dirname(os.path.abspath("f3net_best.pth"))   # cwd
BEST_PATH  = os.path.join(CKPT_DIR, "f3net_best.pth")
FINAL_PATH = os.path.join(CKPT_DIR, "f3net_final.pth")

best_auc      = 0.0
history: list[dict] = []

print("=" * 65)
print(f"  F3Net training  —  {CFG['epochs']} epochs  |  device: {device}")
print("=" * 65)

for epoch in range(1, CFG["epochs"] + 1):
    print(f"\nEpoch {epoch:2d}/{CFG['epochs']}")

    train_loss          = train_one_epoch(train_loader)
    val_acc, val_auc    = validate(val_loader)
    current_lr          = optimizer.param_groups[0]["lr"]

    print(f"  train loss : {train_loss:.5f}")
    print(f"  val acc    : {val_acc*100:.2f}%")
    print(f"  val AUC    : {val_auc:.5f}")
    print(f"  lr         : {current_lr:.2e}")

    history.append({
        "epoch":      epoch,
        "train_loss": train_loss,
        "val_acc":    val_acc,
        "val_auc":    val_auc,
        "lr":         current_lr,
    })

    # ── Checkpoint: save best ─────────────────────────────────────────────
    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), BEST_PATH)
        print(f"  ✓ New best AUC {best_auc:.5f} — saved to {BEST_PATH}")

    scheduler.step()
    torch.cuda.empty_cache()

print("\n" + "=" * 65)
print(f"Training complete.  Best val AUC: {best_auc:.5f}")
print("=" * 65)

  F3Net training  —  12 epochs  |  device: cuda

Epoch  1/12


  train:   3%|█▌                                                       | 50/1875 [00:25<07:00,  4.34it/s]

    batch   50/1875 | loss 0.67283


  train:   5%|███                                                     | 102/1875 [00:39<04:55,  6.00it/s]

    batch  100/1875 | loss 0.64380


...

  train:  96%|████████████████████████████████████████████████████▊  | 1802/1875 [08:47<00:21,  3.35it/s]

    batch 1800/1875 | loss 0.43942


  train:  99%|██████████████████████████████████████████████████████▎| 1852/1875 [09:00<00:03,  7.05it/s]

    batch 1850/1875 | loss 0.43937


  train loss : 0.43961
  val acc    : 81.94%
  val AUC    : 0.88629
  lr         : 2.69e-06
  ✓ New best AUC 0.88629 — saved to /workspace/f3net_best.pth

Training complete.  Best val AUC: 0.88629


## 11. Save Final Model & Training History

In [22]:
# ── Save final weights ────────────────────────────────────────────────────────
torch.save(model.state_dict(), FINAL_PATH)
print(f"Final model saved  : {FINAL_PATH}")
print(f"Best model saved   : {BEST_PATH}  (AUC {best_auc:.5f})")

# ── Save training history to CSV ──────────────────────────────────────────────
history_df = pd.DataFrame(history)
history_csv = os.path.join(CKPT_DIR, "f3net_training_history.csv")
history_df.to_csv(history_csv, index=False)
print(f"Training history   : {history_csv}")

print("\nHistory summary:")
print(history_df.to_string(index=False))

Final model saved  : /workspace/f3net_final.pth
Best model saved   : /workspace/f3net_best.pth  (AUC 0.88629)
Training history   : /workspace/f3net_training_history.csv

History summary:
 epoch  train_loss  val_acc  val_auc       lr
     1    0.519817   0.7876 0.854819 0.000100
     2    0.484676   0.7908 0.862439 0.000098
     3    0.471048   0.7944 0.860292 0.000093
     4    0.464021   0.7998 0.866508 0.000086
     5    0.459329   0.8058 0.870208 0.000075
     6    0.456783   0.8114 0.878449 0.000063
     7    0.454556   0.8122 0.878123 0.000050
     8    0.448803   0.8142 0.879719 0.000038
     9    0.444632   0.8198 0.883882 0.000026
    10    0.441450   0.8166 0.885537 0.000015
    11    0.440772   0.8204 0.884663 0.000008
    12    0.439610   0.8194 0.886291 0.000003


## 12. Final Inference on Validation Set
Load the **best checkpoint**, run inference over the full validation split, and export prediction scores in `[0, 1]` to a CSV.

In [23]:
# ── Load best checkpoint ──────────────────────────────────────────────────────
inference_model = F3Net(dropout=0.0).to(device)   # dropout off for inference
inference_model.load_state_dict(torch.load(BEST_PATH, map_location=device))
inference_model.eval()
print(f"Loaded best model from: {BEST_PATH}")

# ── Run inference ─────────────────────────────────────────────────────────────
all_probs:       list[float] = []
all_preds:       list[int]   = []
all_true_labels: list[int]   = []
all_image_names: list[str]   = []

@torch.no_grad()
def run_inference(loader) -> None:
    for rgb, fft, labels in tqdm(loader, desc="inference"):
        rgb = rgb.to(device, non_blocking=True)
        fft = fft.to(device, non_blocking=True)

        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            logits = inference_model(rgb, fft)

        probs = torch.sigmoid(logits).squeeze(1).cpu().numpy()  # (B,) in [0,1]
        preds = (probs >= 0.5).astype(int)

        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())
        all_true_labels.extend(labels.numpy().tolist())

run_inference(val_loader)

# ── Collect image names for the val split ────────────────────────────────────
for idx in val_idx:
    all_image_names.append(full_dataset.labels.iloc[idx]["image_name"])

# ── Build results DataFrame ───────────────────────────────────────────────────
results_df = pd.DataFrame({
    "image_name":    all_image_names,
    "score":         all_probs,          # probability of AI-generated, in [0,1]
    "predicted":     all_preds,
    "true_label":    all_true_labels,
    "correct":       [p == t for p, t in zip(all_preds, all_true_labels)],
})

# Sort by descending score
results_df = results_df.sort_values("score", ascending=False).reset_index(drop=True)

# ── Save ──────────────────────────────────────────────────────────────────────
out_csv = os.path.join(CKPT_DIR, "f3net_val_predictions.csv")
results_df.to_csv(out_csv, index=False)

final_acc = results_df["correct"].mean()
final_auc = roc_auc_score(results_df["true_label"], results_df["score"])

print(f"\nInference complete — {len(results_df)} samples")
print(f"Accuracy : {final_acc*100:.2f}%")
print(f"ROC-AUC  : {final_auc:.5f}")
print(f"Results  saved to: {out_csv}")
print("\nTop-10 highest-confidence AI detections:")
print(results_df.head(10).to_string(index=False))

Loaded best model from: /workspace/f3net_best.pth


inference: 100%|███████████████████████████████████████████████████████| 105/105 [00:51<00:00,  2.04it/s]



Inference complete — 5000 samples
Accuracy : 81.94%
ROC-AUC  : 0.88629
Results  saved to: /workspace/f3net_val_predictions.csv

Top-10 highest-confidence AI detections:
              image_name    score  predicted  true_label  correct
e478eff1de4c6ca2ea13.jpg 1.000000          1           1     True
849110cc84ba311c788f.jpg 0.999512          1           1     True
e7e82c2cc1a590bdbedd.jpg 0.999512          1           1     True
f5f6f010fdc7b42b57d1.jpg 0.999512          1           1     True
849d0b436277a02d710f.jpg 0.999512          1           1     True
e1037e8aa8b0e37ad8da.jpg 0.999512          1           1     True
714fd51a12d85234d9bc.jpg 0.999023          1           1     True
760524c81bd39af15728.jpg 0.999023          1           1     True
2ae7c50af81c0400c26a.jpg 0.999023          1           1     True
63bdfed124782583cafc.jpg 0.999023          1           1     True


In [24]:
del model
del inference_model
torch.cuda.empty_cache()